# WEEK 5 Questions and Answers

## Apache Spark Fundamentals and DataFrame Operations

This notebook contains the solutions for all Week 5 assignment questions related to Apache Spark DataFrames, data cleaning, filtering, transformation, aggregation, grouping, and data processing pipelines.

In [48]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [49]:
spark = SparkSession.builder \
    .appName("Week5_Question_Solutions") \
    .getOrCreate()

In [50]:
df = spark.read.csv(
    "../data/dataset.csv",
    header=True,
    inferSchema=True
)

df.show(5)

+-------+----------------+------+----------------+-----------+----------+---+------------+--------+-------+--------+--------------+--------+-------------------+
|user_id|transaction_date|region|product_category|sale_amount|      city|age|subscription|  status|  price|store_id|         email|username|      raw_timestamp|
+-------+----------------+------+----------------+-----------+----------+---+------------+--------+-------+--------+--------------+--------+-------------------+
|      1|      2024-05-14|  West|       Furniture|    1829.84|    Mumbai| 54|       Basic|Inactive|1829.84|     104|user1@mail.com|   user1|2024-02-17 09:17:00|
|      2|      2024-08-24|  East|       Furniture|    1771.78|   Kolkata| 55|     Premium|  Active|1771.78|     104|          NULL|   user2|2024-01-07 07:46:00|
|      3|      2024-04-26| North|         Grocery|     343.26|Chandigarh| 59|     Premium|  Active| 343.26|     104|          NULL|   user3|2024-06-19 20:55:00|
|      4|      2024-02-16| North| 

### Question 1

**Q1: What are the key limitations of traditional MapReduce that make Spark a preferred choice for modern big data processing?**

### Answer

Traditional MapReduce processes data by reading from and writing to disk after every stage of execution. This disk-based processing increases execution time, especially for iterative algorithms and interactive analytics.

Apache Spark overcomes these limitations by performing most computations in memory, significantly reducing disk I/O operations. Spark also provides high-level APIs, supports multiple programming languages, and offers built-in libraries for SQL, machine learning, graph processing, and streaming applications.

Because of its speed, simplicity, and scalability, Spark has become one of the most widely used frameworks for modern big data processing.

## Question 2

**Explain how Spark uses In-Memory Computing to speed up iterative machine learning algorithms compared to disk-based systems.**

Apache Spark stores intermediate computation results in memory rather than writing them to disk after every operation. This approach is known as In-Memory Computing.

Machine learning algorithms often require multiple iterations over the same dataset. In disk-based systems like traditional MapReduce, data must be repeatedly read from storage during each iteration, resulting in high latency.

Spark caches data in memory, allowing repeated computations to access the data much faster. This significantly improves execution speed and makes Spark highly efficient for iterative machine learning tasks and large-scale analytics.

## Question 3 

**Write a code snippet to remove all duplicate rows from a DataFrame based on a specific set of columns: user_id and transaction_date.**

In [51]:
# Code Implementation

df = df.dropDuplicates(
    ["user_id", "transaction_date"]
)

df.show()

+-------+----------------+------+----------------+-----------+-----------+---+------------+--------+-------+--------+---------------+--------+-------------------+
|user_id|transaction_date|region|product_category|sale_amount|       city|age|subscription|  status|  price|store_id|          email|username|      raw_timestamp|
+-------+----------------+------+----------------+-----------+-----------+---+------------+--------+-------+--------+---------------+--------+-------------------+
|      1|      2024-05-14|  West|       Furniture|    1829.84|     Mumbai| 54|       Basic|Inactive|1829.84|     104| user1@mail.com|   user1|2024-02-17 09:17:00|
|      2|      2024-08-24|  East|       Furniture|    1771.78|    Kolkata| 55|     Premium|  Active|1771.78|     104|           NULL|   user2|2024-01-07 07:46:00|
|      3|      2024-04-26| North|         Grocery|     343.26| Chandigarh| 59|     Premium|  Active| 343.26|     104|           NULL|   user3|2024-06-19 20:55:00|
|      4|      2024-02

### Observation

Duplicate records were successfully removed based on user_id and transaction_date.

## Question 4

**Given a DataFrame df_sales, write a query to filter for rows where the region is 'West' and then group by product_category to find the average sale_amount.**

In [52]:
df.filter(
    col("region") == "West"
).groupBy(
    "product_category"
).avg(
    "sale_amount"
).show()

+----------------+------------------+
|product_category|  avg(sale_amount)|
+----------------+------------------+
|         Grocery|2234.9605882352944|
|     Electronics|2551.3866666666668|
|        Clothing|          2872.798|
|       Furniture|       2582.450625|
+----------------+------------------+



### Observation

The average sale amount was calculated for each product category in the West region.

## Question 5

**What is the difference between .na.drop() and .na.fill()? Provide a code example of filling null values in a status column with the string 'Unknown'.**

.na.drop() removes rows containing null values.

.na.fill() replaces null values with specified values without removing the rows.

In [53]:
df = df.na.fill(
    {
        "status": "Unknown"
    }
)

df.show()

+-------+----------------+------+----------------+-----------+-----------+---+------------+--------+-------+--------+---------------+--------+-------------------+
|user_id|transaction_date|region|product_category|sale_amount|       city|age|subscription|  status|  price|store_id|          email|username|      raw_timestamp|
+-------+----------------+------+----------------+-----------+-----------+---+------------+--------+-------+--------+---------------+--------+-------------------+
|      1|      2024-05-14|  West|       Furniture|    1829.84|     Mumbai| 54|       Basic|Inactive|1829.84|     104| user1@mail.com|   user1|2024-02-17 09:17:00|
|      2|      2024-08-24|  East|       Furniture|    1771.78|    Kolkata| 55|     Premium|  Active|1771.78|     104|           NULL|   user2|2024-01-07 07:46:00|
|      3|      2024-04-26| North|         Grocery|     343.26| Chandigarh| 59|     Premium|  Active| 343.26|     104|           NULL|   user3|2024-06-19 20:55:00|
|      4|      2024-02

## Question 6

**Write a query to find the total count of records for each city in a DataFrame, but only for cities where the count is greater than 100.**

In [54]:
df.groupBy(
    "city"
).count().filter(
    col("count") > 100
).show()

+----+-----+
|city|count|
+----+-----+
+----+-----+



### Observation

The query identified cities having more than 100 records.

## Question 7 

**How does the immutability of Spark DataFrames affect how you perform "data cleaning" steps like dropping columns or renaming them?**

Spark DataFrames are immutable, which means that once a DataFrame is created, its data cannot be modified directly.

Whenever operations such as dropping a column, renaming a column, filtering records, or updating values are performed, Spark does not change the original DataFrame. Instead, it creates a new DataFrame containing the modified data.

In [55]:
new_df = df.drop("column_name")

In this case, the original DataFrame df remains unchanged, while the result is stored in new_df.

Immutability provides several advantages:

Improves fault tolerance
Prevents accidental modification of original data
Enables Spark to optimize execution plans efficiently
Supports parallel and distributed processing

Therefore, during data cleaning operations, users typically create a new DataFrame after applying transformations.

## Question 8

**Write a Spark command to filter a dataset for rows where the age is between 18 and 30 (inclusive) and the subscription is 'Premium'.**

In [56]:
df.filter(
    (col("age") >= 18) &
    (col("age") <= 30) &
    (col("subscription") == "Premium")
).show()

+-------+----------------+------+----------------+-----------+-----------+---+------------+--------+-------+--------+----------------+--------+-------------------+
|user_id|transaction_date|region|product_category|sale_amount|       city|age|subscription|  status|  price|store_id|           email|username|      raw_timestamp|
+-------+----------------+------+----------------+-----------+-----------+---+------------+--------+-------+--------+----------------+--------+-------------------+
|     17|      2024-09-25|  West|     Electronics|    2536.85|       Pune| 30|     Premium|Inactive|2536.85|     103| user17@mail.com|  user17|2024-07-16 13:03:00|
|     30|      2024-05-13|  East|       Furniture|    1813.97|Bhubaneswar| 19|     Premium|Inactive|1813.97|     103| user30@mail.com|  user30|2024-06-18 17:29:00|
|     36|      2024-10-18|  West|         Grocery|    2251.18|       Pune| 19|     Premium|  Active|2251.18|     102| user36@mail.com|  user36|2024-06-16 05:16:00|
|     43|      2

### Observation

The filtering condition successfully returned only those records where the age was between 18 and 30 years and the subscription type was Premium. This helps analyze a specific customer segment more effectively.

## Question 9 

**When cleaning a dataset, why is it often better to handle null values before performing mathematical aggregations like sum() or avg()?**

Handling null values before performing aggregation operations improves the accuracy and reliability of analytical results.

If null values are present in important numerical columns, aggregation functions such as sum() and avg() may produce incomplete or misleading results.

Cleaning the dataset before aggregation ensures that calculations are performed on valid data and the generated insights are more meaningful and consistent.

## Question 10

**Write the code to revise a column named raw_timestamp by casting it to a TimestampType and renaming it to event_time.**

In [57]:
df = df.withColumn(
    "raw_timestamp",
    to_timestamp(
        col("raw_timestamp"),
        "dd-MM-yyyy HH:mm"
    )
)

df = df.withColumnRenamed(
    "raw_timestamp",
    "event_time"
)

df.printSchema()

root
 |-- user_id: integer (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- region: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- sale_amount: double (nullable = true)
 |-- city: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- subscription: string (nullable = true)
 |-- status: string (nullable = false)
 |-- price: double (nullable = true)
 |-- store_id: integer (nullable = true)
 |-- email: string (nullable = true)
 |-- username: string (nullable = true)
 |-- event_time: timestamp (nullable = true)



### Observation

The raw_timestamp column was successfully converted into TimestampType and renamed to event_time. This improved data consistency and made the column easier to use for time-based analysis.

## Question 11

**Explain the "Shuffle" process that occurs during a grouping operation. Why is it considered a wide transformation?**

A Shuffle is the process of redistributing data across different partitions in a Spark cluster.

During operations such as groupBy(), Spark must bring records having the same key together before performing aggregation. To achieve this, data is moved between executors and partitions across the cluster.

For example, when grouping data by city, all records belonging to the same city must be collected into the same partition. This movement of data is known as a Shuffle.

A grouping operation is considered a Wide Transformation because the output depends on data from multiple partitions rather than a single partition.

Examples of wide transformations include:

groupBy()
join()
distinct()
reduceByKey()

Since shuffling involves network communication and data transfer, it is relatively expensive and can increase execution time. Therefore, minimizing unnecessary shuffle operations is considered a good Spark optimization practice.

## Question 12
**Write a code snippet that identifies and removes rows where the email column contains null values OR the username is an empty string.**

In [58]:
clean_df = df.filter(
    (col("email").isNotNull()) &
    (col("username") != "")
)

clean_df.show()

+-------+----------------+------+----------------+-----------+-----------+---+------------+--------+-------+--------+---------------+--------+-------------------+
|user_id|transaction_date|region|product_category|sale_amount|       city|age|subscription|  status|  price|store_id|          email|username|         event_time|
+-------+----------------+------+----------------+-----------+-----------+---+------------+--------+-------+--------+---------------+--------+-------------------+
|      1|      2024-05-14|  West|       Furniture|    1829.84|     Mumbai| 54|       Basic|Inactive|1829.84|     104| user1@mail.com|   user1|2024-02-17 09:17:00|
|      5|      2024-07-02| South|        Clothing|     1887.6|  Bengaluru| 45|       Basic| Unknown|   NULL|     104| user5@mail.com|   user5|2024-10-15 08:11:00|
|      6|      2024-11-09|  West|     Electronics|     383.35|     Mumbai| 35|     Premium|  Active| 383.35|     103| user6@mail.com|   user6|2024-08-20 04:54:00|
|      7|      2024-04

### Observation

Records containing null email values or empty usernames were identified and removed. This improved data quality by eliminating incomplete and inconsistent records.

## Question 13

**How do you use the .agg() function to calculate multiple statistics at once, such as the min, max, and mean of the price column?**

In [59]:
df.agg(
    min("price").alias("Minimum Price"),
    max("price").alias("Maximum Price"),
    avg("price").alias("Average Price")
).show()

+-------------+-------------+------------------+
|Minimum Price|Maximum Price|     Average Price|
+-------------+-------------+------------------+
|        126.7|      4972.54|2597.4005426356584|
+-------------+-------------+------------------+



### Observation

The agg() function successfully calculated the minimum, maximum, and average values of the price column in a single operation, providing a concise statistical summary of the dataset.

## Question 14
**In the context of cleaning a dataset, what is the risk of using inferSchema=true when your source data contains messy or inconsistent date formats?**

The inferSchema=True option allows Spark to automatically detect the data type of each column while loading a dataset.

Although this feature is convenient, it can cause problems when the source data contains inconsistent or messy date formats.

For example:
```text
12-05-2024
2024/05/12
May 12, 2024
```
All three values represent dates but use different formats.

In such situations, Spark may:

Interpret some date values as strings
Fail to recognize valid dates
Produce null values during conversion
Infer incorrect data types
Generate inconsistent schemas

These issues can affect data quality and lead to incorrect analytical results.

Therefore, it is recommended to clean and standardize date formats before loading the data or explicitly define the schema instead of relying completely on inferSchema=True

## Question 15

**Write a final processing pipeline that:**

**Filters out duplicates.** 

**Fills null prices with 0.** 

**Groups by store_id to calculate total revenue.**

In [60]:
pipeline_df = (
    df
    .dropDuplicates()
    .na.fill({"price": 0})
    .groupBy("store_id")
    .agg(
        sum("sale_amount").alias("Total Revenue")
    )
)

pipeline_df.show()

+--------+-----------------+
|store_id|    Total Revenue|
+--------+-----------------+
|     101|99535.63000000002|
|     103|89602.97000000002|
|     102|69823.98000000001|
|     105|71314.86000000002|
|     104|81439.40999999999|
+--------+-----------------+



### Observation

The complete processing pipeline successfully combined duplicate removal, null value handling, grouping, and aggregation operations. This demonstrated an efficient end-to-end Spark DataFrame workflow for data processing and analysis.

# Conclusion

All Week 5 questions were successfully solved using Apache Spark DataFrame operations. The notebook covers Spark fundamentals, data cleaning, filtering, transformation, aggregation, grouping, and pipeline creation.